## Step 1: Install Feast

Installing the Feast SDK, which will let us define feature stores, register
feature definitions, and fetch features for both training and inference.

In [2]:
!pip install feast --break-system-packages --quiet

Confirming the installation by checking the installed Feast version.

In [3]:
!feast version

Feast SDK Version: "0.64.0"


## Task 1: Initialize the Feast Feature Repository

`feast init` scaffolds a new Feast repository with a default configuration.
This creates a folder called `iris_feature_repo` containing a starter
`feature_store.yaml`, along with some example files we'll clean up next.

In [4]:
!feast init iris_feature_repo


Creating a new Feast repository in /home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo.



Moving into the newly created repository folder and inspecting its contents.

In [5]:
import os
os.chdir("iris_feature_repo")
!ls

README.md  __init__.py	feature_repo


This version of Feast nests the actual repository one level deeper, inside a
`feature_repo` subfolder. Checking what's inside it.

In [6]:
!ls feature_repo

__init__.py  feature_definitions.py  iris_features.py
__pycache__  feature_store.yaml      iris_raw.csv
data	     iris_feature_repo	     test_workflow.py


Moving into `feature_repo`, which is where all our Feast commands and
definitions will actually live from this point forward.

In [7]:
os.chdir("feature_repo")
!ls

__init__.py  feature_definitions.py  iris_features.py
__pycache__  feature_store.yaml      iris_raw.csv
data	     iris_feature_repo	     test_workflow.py


Removing Feast's auto-generated example files (`feature_definitions.py` and
`test_workflow.py`), since we'll write our own IRIS-specific definitions
instead of using the demo "driver" example Feast ships by default.

In [8]:
!rm -f feature_definitions.py
!rm -f test_workflow.py

Confirming the repository is configured with a **local provider** and a
**SQLite online store** — no cloud backend needed for this assignment, since
Task 1 only requires a local backend.

In [9]:
!cat feature_store.yaml

project: iris_feature_repo
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth


Double-checking the current working directory before proceeding, to make sure
all file paths used from here on are relative to the correct folder.

In [10]:
import os
print("Current folder:", os.getcwd())

Current folder: /home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo/feature_repo


## Task 2: Define Entities, Data Sources & Feature Views

Locating the IRIS dataset on disk so it can be copied into this feature
repository.

In [11]:
import subprocess
result = subprocess.run(["find", "/home", "-iname", "iris.csv"], capture_output=True, text=True)
print(result.stdout)

/home/jupyter/iris.csv
/home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/data/iris.csv



Copying the IRIS dataset into the feature repository as `iris_raw.csv`.

In [12]:
import shutil
shutil.copy("/home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/data/iris.csv", "iris_raw.csv")
print("Copied successfully")

Copied successfully


Verifying the copy succeeded and previewing the raw data before any changes
are made to it.

In [13]:
import pandas as pd
df_check = pd.read_csv("iris_raw.csv")
print(df_check.shape)
df_check.head()

(180, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Feast requires every row to have a timestamp, since it's designed for data
that changes over time. IRIS is a static dataset with no natural timestamp, so
a placeholder `event_timestamp` (set to the current time) is added for every
row. An `iris_id` column is also added — a unique integer per row — to serve
as the entity key that uniquely identifies each flower sample. The result is
saved as both CSV and Parquet, since Feast's `FileSource` expects Parquet.

In [14]:
from datetime import datetime

df = pd.read_csv("iris_raw.csv")
df["iris_id"] = range(len(df))
df["event_timestamp"] = datetime.now()

os.makedirs("data", exist_ok=True)
df.to_csv("data/iris.parquet.csv", index=False)
df.to_parquet("data/iris.parquet")

print("Done. Rows:", len(df))
df.head()

Done. Rows: 180


,sepal_length,sepal_width,petal_length,petal_width,species,iris_id,event_timestamp
0,5.1,3.5,1.4,0.2,setosa,0,2026-07-05 07:45:13.003316
1,4.9,3.0,1.4,0.2,setosa,1,2026-07-05 07:45:13.003316
2,4.7,3.2,1.3,0.2,setosa,2,2026-07-05 07:45:13.003316
3,4.6,3.1,1.5,0.2,setosa,3,2026-07-05 07:45:13.003316
4,5.0,3.6,1.4,0.2,setosa,4,2026-07-05 07:45:13.003316


Writing the feature definitions to a file called `iris_features.py`. This
must exist as an actual `.py` file on disk (not just notebook variables),
because `feast apply` scans `.py` files in the repository to discover
definitions.

This file defines three things:
- **Entity** (`iris_id`) — uniquely identifies each iris sample.
- **Data source** (`FileSource`) — points to `data/iris.parquet` and specifies
  `event_timestamp` as the timestamp field.
- **Feature view** (`iris_features`) — maps the four numeric IRIS columns
  (sepal/petal length and width) to the entity and data source above.

In [15]:
%%writefile iris_features.py
from datetime import timedelta
from feast import Entity, FeatureView, FileSource, Field
from feast.types import Float32

# 1. Entity: uniquely identifies each iris sample
iris_entity = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each iris flower sample",
)

# 2. Data source: points to our parquet file
iris_source = FileSource(
    path="data/iris.parquet",
    timestamp_field="event_timestamp",
)

# 3. Feature view: maps the feature columns to the entity and data source
iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris_entity],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
    ],
    source=iris_source,
)

Overwriting iris_features.py


## Task 3: Apply Definitions & Materialize Features

Registering the feature definitions with Feast's registry using `feast apply`.

**Note:** This first attempt failed with a `ConflictingFeatureViewNames` error.
This happened because an earlier accidental re-run of `feast init` created a
nested duplicate copy of the repository, so Feast found two definitions of
`iris_features` and refused to apply either. This was fixed by deleting the
nested duplicate folder before re-running `feast apply` successfully (see
Errors Encountered in the README for details).

In [16]:
!feast apply

/home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo/feature_repo/iris_feature_repo/feature_repo/iris_feature_repo/feature_repo/iris_features.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris_entity = Entity(
/home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo/feature_repo/iris_feature_repo/feature_repo/iris_features.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris_entity = Entity(
/home/jupyter/24f2000612_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_repo/feature_repo/iris_features.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'iris_id'.
  iris_entity = Entity(
No project found in the repository. Using project name iris_feature_repo defined in feature_store.yaml
Applying changes for project iris_feature_repo
Trac

Materializing feature values into the online store — this pushes the actual
feature data from the offline source into SQLite, making it available for
fast, low-latency lookups later (used in Task 5).

In [17]:
from datetime import datetime
now = datetime.now().strftime("%Y-%m-%dT%H:%M:%S")
!feast materialize-incremental {now}

Materializing 1 feature views to 2026-07-05 07:45:23+00:00 into the sqlite online store.

iris_features from 2026-07-05 07:10:09+00:00 to 2026-07-05 07:45:23+00:00:


Checking what materialization produced. Alongside our IRIS files, Feast's
default `init` template also included leftover demo files
(`driver_stats.parquet`, `driver_quality_labels.parquet`) from its "driver"
example project — these aren't related to IRIS and will be removed next.

In [18]:
!ls data/

driver_quality_labels.parquet  iris.parquet	 online_store.db
driver_stats.parquet	       iris.parquet.csv  registry.db


Removing the unrelated demo files so the repository stays focused on the IRIS
feature store.

In [19]:
import os
for f in ["driver_stats.parquet", "driver_quality_labels.parquet"]:
    if os.path.exists(f"data/{f}"):
        os.remove(f"data/{f}")
        print(f"Removed {f}")

Removed driver_stats.parquet
Removed driver_quality_labels.parquet


Confirming the cleanup — only IRIS-related files and Feast's own
`registry.db` / `online_store.db` remain.

In [20]:
!ls data/

iris.parquet  iris.parquet.csv	online_store.db  registry.db


## Task 4: Fetch Features for Training (Offline Store)

Connecting to the feature store and using `get_historical_features` to fetch
feature values for every `iris_id` — this is the key behavioral change
required by this task: training data now comes from Feast's offline store,
not a direct CSV read.

In [21]:
from feast import FeatureStore

store = FeatureStore(repo_path=".")

entity_df = pd.read_csv("data/iris.parquet.csv")[["iris_id", "event_timestamp", "species"]]
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"])

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
).to_df()

print("Fetched from Feast offline store:")
training_df.head()

Fetched from Feast offline store:


,iris_id,event_timestamp,species,sepal_length,sepal_width,petal_length,petal_width
0,0,2026-07-05 07:45:13.003316+00:00,setosa,5.1,3.5,1.4,0.2
1,115,2026-07-05 07:45:13.003316+00:00,virginica,6.4,3.2,5.3,2.3
2,116,2026-07-05 07:45:13.003316+00:00,virginica,6.5,3.0,5.5,1.8
3,117,2026-07-05 07:45:13.003316+00:00,virginica,7.7,3.8,6.7,2.2
4,118,2026-07-05 07:45:13.003316+00:00,virginica,7.7,2.6,6.9,2.3


**Error encountered:** Attempting to train the model failed with a numpy/
scikit-learn binary-incompatibility error. Installing Feast had pulled in a
newer numpy version than scikit-learn was originally compiled against,
leaving a partially broken install. Uninstalling scikit-learn and manually
removing leftover package folders to clear any corrupted files.

In [22]:
!pip uninstall -y scikit-learn --quiet
!rm -rf /opt/micromamba/lib/python3.12/site-packages/sklearn
!rm -rf /opt/micromamba/lib/python3.12/site-packages/scikit_learn*

/opt/micromamba/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=19758) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Reinstalling scikit-learn fresh, with no cache, so pip selects a version
compatible with the numpy version Feast requires.

In [23]:
!pip install --no-cache-dir --break-system-packages scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 20.3 MB/s  0:00:00 eta 0:00:01


Verifying the reinstalled versions. This check alone isn't enough, though —
the Jupyter kernel had already cached the broken module in memory, so the
next step (restarting the kernel) was necessary before training would work.

In [24]:
!python3 -c "import numpy, sklearn; print('numpy:', numpy.__version__); print('sklearn:', sklearn.__version__)"

numpy: 2.2.6
sklearn: 1.9.0


Training the model using the data fetched from Feast's offline store in the
earlier cell. After restarting the kernel to clear the cached broken
scikit-learn state, training completed successfully with 100% evaluation
accuracy, and the model was saved as `model_feast.joblib`.

In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
import joblib

X = training_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = training_df["species"]

X_train, X_eval, y_train, y_eval = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = DecisionTreeClassifier(max_depth=3, random_state=1)
model.fit(X_train, y_train)

acc = metrics.accuracy_score(y_eval, model.predict(X_eval))
print(f"Eval accuracy: {acc:.3f}")

joblib.dump(model, "model_feast.joblib")

Eval accuracy: 1.000


['model_feast.joblib']

## Task 5: Fetch Features for Inference (Online Store)

Simulating a real-time prediction request: given specific iris sample IDs,
fetch their features from the **online** store (not the CSV, not the offline
store) — this is what a production serving system would do.

Fetching features for two sample IDs (5 and 100) directly from the online
store.

In [26]:
sample_ids = [5, 100]

online_features = store.get_online_features(
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
    entity_rows=[{"iris_id": sid} for sid in sample_ids],
).to_dict()

print("Fetched from ONLINE store:")
for i, sid in enumerate(sample_ids):
    print(f"iris_id={sid}: sepal_length={online_features['sepal_length'][i]}, "
          f"sepal_width={online_features['sepal_width'][i]}, "
          f"petal_length={online_features['petal_length'][i]}, "
          f"petal_width={online_features['petal_width'][i]}")

Fetched from ONLINE store:
iris_id=5: sepal_length=5.400000095367432, sepal_width=3.9000000953674316, petal_length=1.7000000476837158, petal_width=0.4000000059604645
iris_id=100: sepal_length=6.300000190734863, sepal_width=3.299999952316284, petal_length=6.0, petal_width=2.5


Loading the trained model and running predictions using the features just
retrieved from the online store.

In [27]:
model_loaded = joblib.load("model_feast.joblib")

X_online = pd.DataFrame({
    "sepal_length": online_features["sepal_length"],
    "sepal_width": online_features["sepal_width"],
    "petal_length": online_features["petal_length"],
    "petal_width": online_features["petal_width"],
})

predictions = model_loaded.predict(X_online)
for sid, pred in zip(sample_ids, predictions):
    print(f"Predicted species for iris_id {sid}: {pred}")

Predicted species for iris_id 5: setosa
Predicted species for iris_id 100: virginica


Verifying training/serving consistency — the whole point of this task. Cross-
checking the online store's feature values against the raw dataset for the
same two sample IDs. The values match exactly, and the predicted species
(`setosa` for ID 5, `virginica` for ID 100) match the true labels — confirming
Feast serves identical features whether accessed offline or online.

In [28]:
check_df = pd.read_csv("data/iris.parquet.csv")
check_df[check_df["iris_id"].isin(sample_ids)][
    ["iris_id", "sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
]

,iris_id,sepal_length,sepal_width,petal_length,petal_width,species
5,5,5.4,3.9,1.7,0.4,setosa
100,100,6.3,3.3,6.0,2.5,virginica
